In [ ]:
import os
import re
from Bio import Entrez, SeqIO
import xml.etree.ElementTree as ET
import json5
import glob
import pandas as pd
import regex as re
import shutil
from collections import defaultdict

Entrez.email = 'A.N.Other@example.com'

def get_ipg(protein_id):
    try:
        search_handle = Entrez.esearch(db="protein", term=protein_id)
        search_results = Entrez.read(search_handle)
        search_handle.close()

        protein_id = search_results["IdList"][0]
        fetch_handle = Entrez.efetch(db="protein", id=protein_id, rettype="ipg", retmode="xml")
        ipg = fetch_handle.read()
        fetch_handle.close()
        
        root = ET.fromstring(ipg)
        protein = root.find(".//Product")
        if protein is not None and protein.get("name"):
            return protein.get("name")

    except Exception as e:
        return f"An error occurred: {e}"

def fetch_fasta(accession, output_path="../tmp/new.fa"):
    try:
        search_handle = Entrez.esearch(db="protein", term=f"{accession}[Locus Tag] OR {accession}[Accession]")
        search_results = Entrez.read(search_handle)
        search_handle.close()

        for protein_id in search_results.get("IdList"):
            fetch_handle = Entrez.efetch(db="protein", id=protein_id, rettype="gb", retmode="text")
            protein = SeqIO.read(fetch_handle, "genbank")
            fetch_handle.close()
    
            with open(output_path, "a") as file:
                file.write('\n')
                SeqIO.write(protein, file, "fasta")
    
            nucleotide = ''
            for feature in protein.features:
                if feature.type == 'CDS':
                    coded_by = feature.qualifiers.get("coded_by", [""])[0]
                    match = re.search(r"([A-Za-z0-9_.]+):", coded_by)
                    if match:
                        nucleotide = match.group(1)
                if feature.type == 'CDS':
                    locus_tag = feature.qualifiers.get("locus_tag", [""])[0]
                    if locus_tag:
                        locus_tag = locus_tag
                        
            if nucleotide:
                print(f"{locus_tag} | {protein.id} -> https://www.ncbi.nlm.nih.gov/nuccore/{nucleotide.split('.')[0]}")

    except Exception as e:
        return f"An error occurred: {e}"

In [ ]:
for i in ['VVMO6_00057', 'VVMO6_00056']:
    if i:
        record = fetch_fasta(i)

In [ ]:
dups = set()
reference = []
with open('../reference/reference.fasta') as f:
    for record in SeqIO.parse(f, 'fasta'):
        if record.seq not in dups and record.id not in dups:
            dups.add(record.seq)
            dups.add(record.id)
            reference.append(record)
        else:
            if record.id not in dups:
                print(record.id)
                reference.append(record)

url = None
with open('../tmp/new.fa') as f:
    a = f.read(1)
    if a == '>':
        f.seek(0)
    
    for record in SeqIO.parse(f, 'fasta'):
        if record.id not in dups and record.seq not in dups:
            if record.id[:3] == 'sp|':
                i = record.id.split('|')[1]
            else:
                i = record.id
            print(f'\t\t\t"{record.id}": "", // {record.description.split(" ", 1)[-1].split('MULTISPECIES: ')[-1]} | {get_ipg(i)}')
            reference.append(record)
            dups.add(record.seq)
            dups.add(record.id)

with open('../tmp/new.fa', 'w') as f:
    pass

with open('../reference/reference.fasta', 'w') as handle:
    SeqIO.write(reference, handle, "fasta")

In [ ]:
%run z1-check.ipynb